# 13 — Final End-to-End Inference and Production Test

## Purpose
Verify that the frozen Plan B decision-engine package from Notebook 12 can be loaded and used for a new bridge-project inference case without retraining, tuning, refitting, or modification.

## Inference chain
```text
New bridge input
    ↓
Frozen Bauwerksart classifier
    ↓
Candidate bridge types + probabilities
    ↓
Frozen zustandsnote condition model
    ↓
Condition evidence for candidates
    ↓
Engineering decision support
```

`laenge`, `breite`, FEM/InfoCAD results, structural dimensioning, reinforcement design and load-bearing verification are outside the ML inference contract.


In [1]:
# 01 — Locate the frozen production package from Notebook 12

from pathlib import Path
import os, hashlib, json
import numpy as np
import pandas as pd
import joblib

def find_project_root():
    env_root = os.getenv("BRIDGE_PROJECT_ROOT")
    if env_root:
        root = Path(env_root).expanduser().resolve()
        if (root / "Dataset_PlanA-B").exists():
            return root
        raise FileNotFoundError(f"BRIDGE_PROJECT_ROOT does not contain Dataset_PlanA-B: {root}")
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "Dataset_PlanA-B").exists():
            return candidate
    raise FileNotFoundError("Project root not found. Set BRIDGE_PROJECT_ROOT.")

PROJECT_ROOT = find_project_root()
DATASET_ROOT = PROJECT_ROOT / "Dataset_PlanA-B"
OUTPUT_ROOT = PROJECT_ROOT / "Output_PlanA-B"

MODEL_DIR = OUTPUT_ROOT / "12_Final_ML_Model_Freeze_and_Packaging" / "model_package"
OUTPUT_DIR = OUTPUT_ROOT / "13_Final_End_to_End_Inference_and_Production_Test"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CLASSIFIER_PATH = MODEL_DIR / "bridge_type_classifier.joblib"
CONDITION_PATH = MODEL_DIR / "condition_model.joblib"
MODEL_MANIFEST_PATH = MODEL_DIR / "model_manifest.json"

required_files = [CLASSIFIER_PATH, CONDITION_PATH, MODEL_MANIFEST_PATH]
missing = [str(p) for p in required_files if not p.exists()]
if missing:
    raise FileNotFoundError("Missing frozen Notebook 12 package files:\n" + "\n".join(missing))

print("Project root:", PROJECT_ROOT)
print("Frozen model package:", MODEL_DIR)
print("Production-test output:", OUTPUT_DIR)
print("Frozen package: FOUND")


Project root: C:\Datenanalyse\final Project
Frozen model package: C:\Datenanalyse\final Project\Output_PlanA-B\12_Final_ML_Model_Freeze_and_Packaging\model_package
Production-test output: C:\Datenanalyse\final Project\Output_PlanA-B\13_Final_End_to_End_Inference_and_Production_Test
Frozen package: FOUND


## 00A — DATA SOURCE / INPUT–OUTPUT MANIFEST

| Item | Source | Transfer | Role | Destination |
|---|---|---|---|---|
| Frozen classifier | Notebook 12 `model_package/bridge_type_classifier.joblib` | Local read | Bauwerksart inference | In-memory model |
| Frozen condition model | Notebook 12 `model_package/condition_model.joblib` | Local read | Zustandsnote evidence | In-memory model |
| Frozen contract | Notebook 12 `model_manifest.json` | Local JSON read | Contract verification | In-memory manifest |
| New bridge input | Notebook 13 | In-memory | Production-test input | `NEW_BRIDGE` |
| Inference outputs | Notebook 13 | Local write | Production-test artifacts | `Output_PlanA-B/13_...` |
| Manifest/inventory | Notebook 13 | Local write | Provenance/audit | JSON/TXT/CSV |

### Transfer chain
```text
Notebook 12 frozen package
        ↓
Notebook 13 contract + hash verification
        ↓
new bridge input
        ↓
classifier inference
        ↓
condition evidence
        ↓
production-test package
```

No BASt/DWD/Traffic download, retraining, FEM, InfoCAD or structural design occurs here.


In [2]:
# 02 — Verify the Notebook 12 frozen manifest

manifest = json.loads(MODEL_MANIFEST_PATH.read_text(encoding="utf-8"))

EXPECTED_INPUTS = ["latitude", "longitude", "dtv", "bauwerkstoff"]

assert manifest.get("stage") == 12
assert manifest.get("status") == "FROZEN"
assert manifest.get("package_scope") == "Plan B bridge-type decision engine"
assert manifest.get("classifier", {}).get("target") == "bauwerksart"
assert manifest.get("classifier", {}).get("features") == EXPECTED_INPUTS
assert manifest.get("condition_model", {}).get("target") == "zustandsnote"
assert manifest.get("condition_model", {}).get("features") == EXPECTED_INPUTS + ["bauwerksart"]

contract = manifest.get("model_contract", {})
assert contract.get("length_used_by_ml") is False
assert contract.get("width_used_by_ml") is False
assert contract.get("fem_input_to_ml") is False
assert contract.get("structural_design") is False

print("Frozen manifest: PASS")
print("Source package stage: 12")
print("Input contract: latitude + longitude + dtv + bauwerkstoff")


Frozen manifest: PASS
Source package stage: 12
Input contract: latitude + longitude + dtv + bauwerkstoff


In [3]:
# 03 — Verify frozen package integrity

def sha256(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

integrity = pd.DataFrame([
    {"file": p.name, "path": str(p), "size_bytes": p.stat().st_size, "sha256": sha256(p)}
    for p in required_files
])
display(integrity)
assert (integrity["size_bytes"] > 0).all()
print("Package integrity check: PASS")


,file,path,size_bytes,sha256
0,bridge_type_classifier.joblib,C:\Datenanalyse\final Project\Output_PlanA-B\1...,13430703187,28fff172bdbbab2de96b5273f3c02d62d09f0ac5e790f0...
1,condition_model.joblib,C:\Datenanalyse\final Project\Output_PlanA-B\1...,350505539,e47637fbe76018ec76fc2852fedb583439e56077f72fc9...
2,model_manifest.json,C:\Datenanalyse\final Project\Output_PlanA-B\1...,1146,ff6a18736fc412023fe517dc8e9fd31a1154de6833ac29...


Package integrity check: PASS


In [4]:
# 04 — Load frozen models only

# Do not use mmap_mode: these joblib artifacts are packaged sklearn objects
# and should be loaded normally for a deterministic production smoke test.
classifier = joblib.load(CLASSIFIER_PATH)
condition_model = joblib.load(CONDITION_PATH)

CLS_FEATURES = list(manifest["classifier"]["features"])
REG_FEATURES = list(manifest["condition_model"]["features"])

assert CLS_FEATURES == EXPECTED_INPUTS
assert REG_FEATURES == EXPECTED_INPUTS + ["bauwerksart"]
assert hasattr(classifier, "predict_proba")
assert hasattr(condition_model, "predict")

print("Classifier load: PASS")
print("Condition model load: PASS")
print("No retraining performed.")


Classifier load: PASS
Condition model load: PASS
No retraining performed.


In [5]:
# 05 — Define one independent new-bridge inference case

NEW_BRIDGE = pd.DataFrame([{
    "latitude": 50.7374,
    "longitude": 7.0982,
    "dtv": 25000.0,
    "bauwerkstoff": "Stahlbeton",
}])

assert list(NEW_BRIDGE.columns) == EXPECTED_INPUTS
NEW_BRIDGE["latitude"] = pd.to_numeric(NEW_BRIDGE["latitude"], errors="coerce")
NEW_BRIDGE["longitude"] = pd.to_numeric(NEW_BRIDGE["longitude"], errors="coerce")
NEW_BRIDGE["dtv"] = pd.to_numeric(NEW_BRIDGE["dtv"], errors="coerce")
NEW_BRIDGE["bauwerkstoff"] = NEW_BRIDGE["bauwerkstoff"].astype("string").str.strip()

assert NEW_BRIDGE["latitude"].between(-90, 90).all()
assert NEW_BRIDGE["longitude"].between(-180, 180).all()
assert NEW_BRIDGE["dtv"].ge(0).all()
assert NEW_BRIDGE["bauwerkstoff"].notna().all()
assert NEW_BRIDGE["bauwerkstoff"].ne("").all()

display(NEW_BRIDGE)
print("New-bridge input validation: PASS")


,latitude,longitude,dtv,bauwerkstoff
0,50.7374,7.0982,25000.0,Stahlbeton


New-bridge input validation: PASS


In [6]:
# 06 — Final classifier inference

proba = classifier.predict_proba(NEW_BRIDGE[CLS_FEATURES])[0]
classes = classifier.named_steps["model"].classes_

classification = pd.DataFrame({
    "bauwerksart": classes,
    "classification_probability": proba,
}).sort_values("classification_probability", ascending=False).reset_index(drop=True)

classification["classification_probability_%"] = (
    classification["classification_probability"] * 100
).round(2)

TOP_K = min(10, len(classification))
top_candidates = classification.head(TOP_K).copy()

display(top_candidates)
assert np.isfinite(classification["classification_probability"]).all()
assert np.isclose(classification["classification_probability"].sum(), 1.0)

print("Classifier inference: PASS")
print("Candidate count:", len(classification))


,bauwerksart,classification_probability,classification_probability_%
0,"Plattenbalkenbrücke, Trägerrostbrücke",0.236000,23.60
1,Gewölbe- bzw. Bogenbrücke,0.110000,11.00
2,Plattenbrücke,0.100000,10.00
3,Balkenbrücke / Mittelträger / Trapezplatte,0.096000,9.60
4,Brücke mit Balken- / Plattenmischsystem,0.086000,8.60
5,Bogenbrücke mit Bogenscheiben,0.066000,6.60
6,Hohlkastenbrücke,0.060000,6.00
7,"Rohr als Brücke, ohne Ummantelung",0.058000,5.80
8,Gewölbe-/Bogenbrücke ohne Aufbeton,0.047672,4.77
9,Brücke als geschlossener Rahmen,0.030328,3.03


Classifier inference: PASS
Candidate count: 43


In [7]:
# 07 — Condition evidence for candidate types

condition_input = pd.DataFrame({
    "latitude": [NEW_BRIDGE.loc[0, "latitude"]] * TOP_K,
    "longitude": [NEW_BRIDGE.loc[0, "longitude"]] * TOP_K,
    "dtv": [NEW_BRIDGE.loc[0, "dtv"]] * TOP_K,
    "bauwerkstoff": [NEW_BRIDGE.loc[0, "bauwerkstoff"]] * TOP_K,
    "bauwerksart": top_candidates["bauwerksart"].tolist(),
})

condition_values = np.clip(
    condition_model.predict(condition_input[REG_FEATURES]),
    1, 4,
)

final_candidates = top_candidates.copy()
final_candidates["predicted_zustandsnote_evidence"] = np.round(condition_values, 3)

display(final_candidates)
print("Condition evidence inference: PASS")
print("Condition evidence is separate from classifier probability.")


,bauwerksart,classification_probability,classification_probability_%,predicted_zustandsnote_evidence
0,"Plattenbalkenbrücke, Trägerrostbrücke",0.236000,23.60,2.389
1,Gewölbe- bzw. Bogenbrücke,0.110000,11.00,2.153
2,Plattenbrücke,0.100000,10.00,2.110
3,Balkenbrücke / Mittelträger / Trapezplatte,0.096000,9.60,2.073
4,Brücke mit Balken- / Plattenmischsystem,0.086000,8.60,2.266
5,Bogenbrücke mit Bogenscheiben,0.066000,6.60,2.310
6,Hohlkastenbrücke,0.060000,6.00,2.422
7,"Rohr als Brücke, ohne Ummantelung",0.058000,5.80,1.944
8,Gewölbe-/Bogenbrücke ohne Aufbeton,0.047672,4.77,2.173
9,Brücke als geschlossener Rahmen,0.030328,3.03,2.045


Condition evidence inference: PASS
Condition evidence is separate from classifier probability.


In [8]:
# 08 — End-to-end architecture gate

assert list(NEW_BRIDGE.columns) == EXPECTED_INPUTS
assert "bauwerksart" not in NEW_BRIDGE.columns
assert "zustandsnote" not in NEW_BRIDGE.columns

print("Architecture gate: PASS")
print("Forbidden structural variables entered into classifier: NONE")
print("Target leakage through bauwerksart: NONE")
print("Target leakage through zustandsnote: NONE")
print("FEM / InfoCAD input to ML: NONE")


Architecture gate: PASS
Forbidden structural variables entered into classifier: NONE
Target leakage through bauwerksart: NONE
Target leakage through zustandsnote: NONE
FEM / InfoCAD input to ML: NONE


In [9]:
# 09 — Export the production inference result

OUT = Path("outputs") / "bridge_type_selection" / "production_inference"
OUT.mkdir(parents=True, exist_ok=True)

input_path = OUT / "16_new_bridge_input.csv"
classification_path = OUT / "16_classification_candidates.csv"
evidence_path = OUT / "16_condition_evidence.csv"
            
NEW_BRIDGE.to_csv(input_path, index=False, encoding="utf-8-sig")
classification.to_csv(classification_path, index=False, encoding="utf-8-sig")
final_candidates.to_csv(evidence_path, index=False, encoding="utf-8-sig")

production_manifest = {
    "stage": 16,
    "status": "PRODUCTION_INFERENCE_TEST_COMPLETE",
    "model_status": "FROZEN",
    "inputs": EXPECTED_INPUTS,
    "classifier_target": "bauwerksart",
    "condition_evidence": "zustandsnote",
    "structural_design": False,
    "fem_input_to_ml": False,
    "length_used_by_ml": False,
    "width_used_by_ml": False,
    "classifier_file": CLASSIFIER_PATH.name,
    "condition_model_file": CONDITION_PATH.name,
    "top_candidate": str(final_candidates.loc[0, "bauwerksart"]),
    "top_candidate_probability": float(final_candidates.loc[0, "classification_probability"]),
}

(OUT / "16_production_inference_manifest.json").write_text(
    json.dumps(production_manifest, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("Production inference artifacts exported:", OUT)


Production inference artifacts exported: outputs\bridge_type_selection\production_inference


In [10]:
# 10 — Final status

print("16 STATUS: COMPLETE")
print("Frozen classifier: VERIFIED")
print("Frozen condition model: VERIFIED")
print("Four-input interface: VERIFIED")
print("End-to-end inference: VERIFIED")
print("Structural design: EXCLUDED")
print("FEM / InfoCAD: EXCLUDED")
print("Model retraining: NOT PERFORMED")


16 STATUS: COMPLETE
Frozen classifier: VERIFIED
Frozen condition model: VERIFIED
Four-input interface: VERIFIED
End-to-end inference: VERIFIED
Structural design: EXCLUDED
FEM / InfoCAD: EXCLUDED
Model retraining: NOT PERFORMED


In [11]:
# 09 — Export the production inference result

input_path = OUTPUT_DIR / "13_new_bridge_input.csv"
classification_path = OUTPUT_DIR / "13_classification_candidates.csv"
evidence_path = OUTPUT_DIR / "13_condition_evidence.csv"
integrity_path = OUTPUT_DIR / "13_frozen_model_integrity.csv"

NEW_BRIDGE.to_csv(input_path, index=False, encoding="utf-8-sig")
classification.to_csv(classification_path, index=False, encoding="utf-8-sig")
final_candidates.to_csv(evidence_path, index=False, encoding="utf-8-sig")
integrity.to_csv(integrity_path, index=False, encoding="utf-8-sig")

production_manifest = {
    "stage": 13,
    "status": "PRODUCTION_INFERENCE_TEST_COMPLETE",
    "model_status": "FROZEN",
    "source_model_package_stage": 12,
    "model_package_dir": str(MODEL_DIR),
    "inputs": EXPECTED_INPUTS,
    "classifier_target": "bauwerksart",
    "condition_evidence": "zustandsnote",
    "structural_design": False,
    "fem_input_to_ml": False,
    "length_used_by_ml": False,
    "width_used_by_ml": False,
    "classifier_file": CLASSIFIER_PATH.name,
    "condition_model_file": CONDITION_PATH.name,
    "classifier_sha256": sha256(CLASSIFIER_PATH),
    "condition_model_sha256": sha256(CONDITION_PATH),
    "model_manifest_sha256": sha256(MODEL_MANIFEST_PATH),
    "top_candidate": str(final_candidates.loc[0, "bauwerksart"]),
    "top_candidate_probability": float(final_candidates.loc[0, "classification_probability"]),
    "note": "Top candidate is an ML inference result, not a structural design verdict.",
}

MANIFEST_OUT = OUTPUT_DIR / "13_data_manifest.json"
MANIFEST_TXT = OUTPUT_DIR / "13_data_manifest.txt"
INVENTORY_OUT = OUTPUT_DIR / "13_data_inventory.csv"

MANIFEST_OUT.write_text(json.dumps(production_manifest, indent=2, ensure_ascii=False), encoding="utf-8")

inventory = pd.DataFrame([
    {"artifact": p.name, "path": str(p), "exists": p.exists(),
     "size_bytes": p.stat().st_size if p.exists() else None, "role": role}
    for p, role in [
        (input_path, "new bridge input"),
        (classification_path, "classifier candidates"),
        (evidence_path, "condition evidence"),
        (integrity_path, "frozen model hashes"),
        (MANIFEST_OUT, "production inference manifest"),
    ]
])
inventory.to_csv(INVENTORY_OUT, index=False, encoding="utf-8-sig")

MANIFEST_TXT.write_text(
    "Notebook 13 — Final End-to-End Inference and Production Test\n"
    "============================================================\n\n"
    f"PROJECT_ROOT: {PROJECT_ROOT}\n"
    f"SOURCE MODEL PACKAGE: {MODEL_DIR}\n"
    f"OUTPUT_DIR: {OUTPUT_DIR}\n\n"
    "MODEL PACKAGE SOURCE: Notebook 12\n"
    "INPUTS: latitude, longitude, dtv, bauwerkstoff\n"
    "CLASSIFIER TARGET: bauwerksart\n"
    "CONDITION EVIDENCE TARGET: zustandsnote\n"
    "LENGTH/WIDTH AS ML INPUTS: NO\n"
    "FEM/InfoCAD: NO\n"
    "STRUCTURAL DESIGN: NO\n\n"
    f"CLASSIFIER SHA256: {production_manifest['classifier_sha256']}\n"
    f"CONDITION MODEL SHA256: {production_manifest['condition_model_sha256']}\n"
    f"MODEL MANIFEST SHA256: {production_manifest['model_manifest_sha256']}\n\n"
    f"INVENTORY: {INVENTORY_OUT}\n",
    encoding="utf-8",
)

print("Production inference artifacts exported:", OUTPUT_DIR)


Production inference artifacts exported: C:\Datenanalyse\final Project\Output_PlanA-B\13_Final_End_to_End_Inference_and_Production_Test


In [12]:
# 10 — Final status

assert MANIFEST_OUT.exists()
assert INVENTORY_OUT.exists()
assert input_path.exists()
assert classification_path.exists()
assert evidence_path.exists()
assert integrity_path.exists()

print("13 STATUS: COMPLETE")
print("Frozen classifier: VERIFIED")
print("Frozen condition model: VERIFIED")
print("Four-input interface: VERIFIED")
print("End-to-end inference: VERIFIED")
print("Frozen-model hashes: VERIFIED")
print("Structural design: EXCLUDED")
print("FEM / InfoCAD: EXCLUDED")
print("Model retraining: NOT PERFORMED")
print("Output directory:", OUTPUT_DIR)
print("Data manifest:", MANIFEST_TXT)


13 STATUS: COMPLETE
Frozen classifier: VERIFIED
Frozen condition model: VERIFIED
Four-input interface: VERIFIED
End-to-end inference: VERIFIED
Frozen-model hashes: VERIFIED
Structural design: EXCLUDED
FEM / InfoCAD: EXCLUDED
Model retraining: NOT PERFORMED
Output directory: C:\Datenanalyse\final Project\Output_PlanA-B\13_Final_End_to_End_Inference_and_Production_Test
Data manifest: C:\Datenanalyse\final Project\Output_PlanA-B\13_Final_End_to_End_Inference_and_Production_Test\13_data_manifest.txt
